In [1]:
# ================================
# 1. Install Dependencies
# ================================
!pip install unsloth transformers datasets scikit-learn accelerate

# ================================
# 2. Imports
# ================================
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from tqdm import tqdm

# ================================
# 3. Load Your Dataset
# ================================
file_path = "/content/CN_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

print("Total samples:", len(data))

# Convert to HF dataset
dataset = Dataset.from_list(data)

# ================================
# 4. Load LLaMA 3.1 (Unsloth optimized)
# ================================
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ================================
# 5. Define Zero-Shot Prompt
# ================================
def format_prompt(example):
    return f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
"""

# ================================
# 6. Inference Function
# ================================
def predict(example):
    prompt = format_prompt(example)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.0,   # deterministic
            do_sample=False
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only predicted label
    pred = response.split("### Response:")[-1].strip().split()[0]

    return pred

# ================================
# 7. Run Predictions
# ================================
y_true = []
y_pred = []

for example in tqdm(dataset):
    gt = example["output"].strip()
    pred = predict(example)

    y_true.append(gt)
    y_pred.append(pred)

# ================================
# 8. Evaluation Metrics
# ================================
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== Evaluation Results =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_true, y_pred))

# ================================
# 9. Show Sample Predictions
# ================================
for i in range(5):
    print("\n--- Example ---")
    print("Input :", dataset[i]["input"])
    print("GT    :", y_true[i])
    print("Pred  :", y_pred[i])

Total samples: 1952


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  0%|          | 0/1952 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=10) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 1952/1952 [57:55<00:00,  1.78s/it]


===== Evaluation Results =====
Accuracy : 0.3366
Precision: 0.5342
Recall   : 0.3366
F1 Score : 0.2472

===== Classification Report =====
              precision    recall  f1-score   support

    based_on       0.50      0.08      0.13        66
   based_on.       0.00      0.00      0.00         0
  implements       0.33      0.13      0.19        69
    improves       0.73      0.27      0.39        60
 no_relation       0.67      0.06      0.11       914
no_relation,       0.00      0.00      0.00         0
no_relation.       0.00      0.00      0.00         0
     part_of       0.50      0.01      0.02       184
    used_for       0.36      0.87      0.51       657
   used_for,       0.00      0.00      0.00         0
   used_for.       0.00      0.00      0.00         0
    uses_for       0.00      0.00      0.00         2

    accuracy                           0.34      1952
   macro avg       0.26      0.12      0.11      1952
weighted avg       0.53      0.34      0.25      


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/

In [ ]:
!rm -rf /content/unsloth_compiled_cache